In [1]:
!pip install -q aif360 scikit-learn pandas numpy seaborn

import os
import urllib.request
import aif360

aif360_dir = os.path.dirname(aif360.__file__)
raw_dir = os.path.join(aif360_dir, "data", "raw")

files = {
    "german/german.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
    "german/german.doc": "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.doc",
    "compas/compas-scores-two-years.csv": "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
}

for rel_path, url in files.items():
    dest = os.path.join(raw_dir, rel_path)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)

print("Fast setup complete!")

Fast setup complete!


I selected the COMPAS, German Credit and Titanic datasets (Adult required too much computational time) Here I will calculate the demographic parity difference, the disparate ratio, and the consistency from the original labels of the datsets before any mitigation.

In [2]:
import pandas as pd
import seaborn as sns
from aif360.datasets import GermanDataset, CompasDataset, StandardDataset
from aif360.metrics import BinaryLabelDatasetMetric

# Load and clean Titanic dataset (~891 rows) into AIF360 format
titanic_df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]].dropna()
titanic_df["sex"] = (titanic_df["sex"] == "female").astype(int)  # 1 = female (privileged for survival), 0 = male (unprivileged)

titanic_aif = StandardDataset(
    df=titanic_df,
    label_name="survived",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]]
)

datasets = {
    "German Credit (sex)": (GermanDataset(), [{"sex": 1}], [{"sex": 0}]),
    "COMPAS (race)": (CompasDataset(), [{"race": 1}], [{"race": 0}]),
    "Titanic (sex)": (titanic_aif, [{"sex": 1}], [{"sex": 0}])
}

print("=== RAW DATASET METRICS (ORIGINAL LABELS BEFORE MITIGATION) ===\n")
for name, (ds, priv, unpriv) in datasets.items():
    metric = BinaryLabelDatasetMetric(
        ds,
        unprivileged_groups=unpriv,
        privileged_groups=priv
    )
    print(f"{name}:")
    print(f"  Samples:     {ds.features.shape[0]} | Features: {ds.features.shape[1]}")
    print(f"  DP_Diff:     {metric.mean_difference():.4f}")
    print(f"  DIR:         {metric.disparate_impact():.4f}")
    print(f"  Consistency: {metric.consistency()[0]:.4f}\n")

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


=== RAW DATASET METRICS (ORIGINAL LABELS BEFORE MITIGATION) ===

German Credit (sex):
  Samples:     1000 | Features: 58
  DP_Diff:     -0.0748
  DIR:         0.8966
  Consistency: 0.6816

COMPAS (race):
  Samples:     6167 | Features: 401
  DP_Diff:     -0.0971
  DIR:         0.8404
  Consistency: 0.6770

Titanic (sex):
  Samples:     714 | Features: 6
  DP_Diff:     -0.5495
  DIR:         0.2720
  Consistency: 0.7134



Performing the Stratified Split


In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

def stratified_aif360_split(dataset, seed=42):
    """Splits an AIF360 dataset into 60% train, 20% val, 20% test stratified by (label, protected_attribute)."""
    y = dataset.labels.ravel()
    s = dataset.protected_attributes.ravel()
    strata = [f"{int(yi)}_{int(si)}" for yi, si in zip(y, s)]
    indices = np.arange(len(y))

    # 60% Train, 40% Temp
    train_idx, temp_idx = train_test_split(
        indices, test_size=0.40, stratify=strata, random_state=seed
    )
    temp_strata = [strata[i] for i in temp_idx]

    # 20% Validation, 20% Held-out Test
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, stratify=temp_strata, random_state=seed
    )

    return (
        dataset.subset(train_idx),
        dataset.subset(val_idx),
        dataset.subset(test_idx)
    )

splits = {}
print("=== STRATIFIED SPLIT VERIFICATION (60% Train / 20% Val / 20% Test) ===\n")
for name, (ds, priv, unpriv) in datasets.items():
    train_ds, val_ds, test_ds = stratified_aif360_split(ds, seed=42)
    splits[name] = (train_ds, val_ds, test_ds, priv, unpriv)

    print(f"{name}:")
    print(f"  Total: {len(ds.labels)} -> Train: {len(train_ds.labels)} | Val: {len(val_ds.labels)} | Test: {len(test_ds.labels)}")
    print(f"  Positive label rate -> Train: {train_ds.labels.mean():.3f} | Val: {val_ds.labels.mean():.3f} | Test: {test_ds.labels.mean():.3f}\n")

=== STRATIFIED SPLIT VERIFICATION (60% Train / 20% Val / 20% Test) ===

German Credit (sex):
  Total: 1000 -> Train: 600 | Val: 200 | Test: 200
  Positive label rate -> Train: 1.300 | Val: 1.300 | Test: 1.300

COMPAS (race):
  Total: 6167 -> Train: 3700 | Val: 1233 | Test: 1234
  Positive label rate -> Train: 0.455 | Val: 0.456 | Test: 0.455

Titanic (sex):
  Total: 714 -> Train: 428 | Val: 143 | Test: 143
  Positive label rate -> Train: 0.407 | Val: 0.406 | Test: 0.406



In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from aif360.algorithms.postprocessing import RejectOptionClassification

mitigated_outputs = {}

print("=== STEP 4: TRAINING BASE CLASSIFIER & FITTING ROC POST-PROCESSING ===\n")
for name, (train_ds, val_ds, test_ds, priv, unpriv) in splits.items():
    # 1. Standardize features using training statistics only
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_ds.features)
    y_train = train_ds.labels.ravel()

    X_val = scaler.transform(val_ds.features)
    X_test = scaler.transform(test_ds.features)

    # 2. Train standard probabilistic classifier on unmodified training data
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)

    fav_idx = list(clf.classes_).index(train_ds.favorable_label)

    # 3. Store unmitigated baseline predictions for Validation and Test sets
    val_pred_ds = val_ds.copy(deepcopy=True)
    val_pred_ds.scores = clf.predict_proba(X_val)[:, fav_idx].reshape(-1, 1)
    val_pred_ds.labels = clf.predict(X_val).reshape(-1, 1)

    test_pred_ds = test_ds.copy(deepcopy=True)
    test_pred_ds.scores = clf.predict_proba(X_test)[:, fav_idx].reshape(-1, 1)
    test_pred_ds.labels = clf.predict(X_test).reshape(-1, 1)

    # 4. Fit RejectOptionClassification on validation set to learn optimal threshold & margin
    roc = RejectOptionClassification(
        unprivileged_groups=unpriv,
        privileged_groups=priv,
        low_class_thresh=0.01,
        high_class_thresh=0.99,
        num_class_thresh=100,
        num_ROC_margin=50,
        metric_name="Statistical parity difference",
        metric_ub=0.05,
        metric_lb=-0.05
    )
    roc = roc.fit(val_ds, val_pred_ds)

    # 5. Apply ROC mitigation to held-out test set predictions
    test_mitigated_ds = roc.predict(test_pred_ds)

    mitigated_outputs[name] = {
        "test_true": test_ds,
        "test_before": test_pred_ds,
        "test_after": test_mitigated_ds,
        "priv": priv,
        "unpriv": unpriv,
        "class_thresh": roc.classification_threshold,
        "roc_margin": roc.ROC_margin
    }

    print(f"{name}:")
    print(f"  Learned Classification Threshold: {roc.classification_threshold:.4f}")
    print(f"  Learned ROC Margin (theta):       {roc.ROC_margin:.4f}\n")

=== STEP 4: TRAINING BASE CLASSIFIER & FITTING ROC POST-PROCESSING ===

German Credit (sex):
  Learned Classification Threshold: 0.6534
  Learned ROC Margin (theta):       0.1202

COMPAS (race):
  Learned Classification Threshold: 0.5445
  Learned ROC Margin (theta):       0.0465

Titanic (sex):
  Learned Classification Threshold: 0.4852
  Learned ROC Margin (theta):       0.3366



In [5]:
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, f1_score
from aif360.metrics import BinaryLabelDatasetMetric, ClassificationMetric

comparison_rows = []

for name, out in mitigated_outputs.items():
    test_true = out["test_true"]
    priv = out["priv"]
    unpriv = out["unpriv"]
    y_true = test_true.labels.ravel()
    fav_label = test_true.favorable_label

    for stage_label, pred_ds in [
        ("Before Mitigation (Test)", out["test_before"]),
        ("After ROC Mitigation (Test)", out["test_after"])
    ]:
        bld = BinaryLabelDatasetMetric(pred_ds, unprivileged_groups=unpriv, privileged_groups=priv)
        cls = ClassificationMetric(test_true, pred_ds, unprivileged_groups=unpriv, privileged_groups=priv)
        y_pred = pred_ds.labels.ravel()

        comparison_rows.append({
            "Dataset": name,
            "Stage": stage_label,
            "DP_Diff": round(bld.mean_difference(), 4),
            "DIR": round(bld.disparate_impact(), 4),
            "Consistency": round(bld.consistency()[0], 4),
            "Accuracy": round(cls.accuracy(), 4),
            "Balanced_Acc": round(balanced_accuracy_score(y_true, y_pred), 4),
            "F1_Score": round(f1_score(y_true, y_pred, pos_label=fav_label), 4)
        })

results_df = pd.DataFrame(comparison_rows)
print("=== HELD-OUT TEST SET COMPARISON: BEFORE VS. AFTER MITIGATION ===\n")
print(results_df.to_string(index=False))

=== HELD-OUT TEST SET COMPARISON: BEFORE VS. AFTER MITIGATION ===

            Dataset                       Stage  DP_Diff    DIR  Consistency  Accuracy  Balanced_Acc  F1_Score
German Credit (sex)    Before Mitigation (Test)  -0.2308 0.7279       0.7390    0.7100        0.6214    0.8027
German Credit (sex) After ROC Mitigation (Test)   0.0655 1.1253       0.6220    0.6550        0.6631    0.7229
      COMPAS (race)    Before Mitigation (Test)  -0.1698 0.7637       0.8549    0.6653        0.6572    0.7089
      COMPAS (race) After ROC Mitigation (Test)  -0.0217 0.9621       0.8337    0.6621        0.6581    0.6936
      Titanic (sex)    Before Mitigation (Test)  -0.7868 0.1128       0.7133    0.8182        0.8087    0.7719
      Titanic (sex) After ROC Mitigation (Test)   0.0294 1.0707       0.8014    0.6643        0.6574    0.6000
